# D2.9 · The fleet kill switch

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *AI for Security*

Builds on **[D2.8 · Regulatory clock](https://spbreed.github.io/cyber-commons/lessons/D2.8.html)**.

| | |
|---|---|
| Tools used | Vault, Kubernetes |

## What this lesson is

**What it covers.** Kill a fleet, then check what the revoked-credential step changes about what an attacker still holds afterwards.

**Why a security engineer needs it.** Terminating agents while their tokens stay valid leaves the persistence in place. In the incident, third-party access ended when the third party revoked keys — not when the agents stopped. The control it builds is: a tested kill path independent of the agent execution path, snapshot before terminate, revocation in the same action, a measured activation target and named authority to pull it (C8.3).

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The third party's exposure ended when the third party revoked its keys, not when the agents were stopped. Terminating a fleet whose credentials stay valid moves the incident rather than ending it.

> **At CyberTravels.** Terminating CyberTravels' four agents while their bearer tokens stay valid moves the incident rather than ending it. R5.

## 2 · The framework

```
   one selector, one action, in this order

      snapshot state + transcripts     <- or the incident is unreconstructable
              |
           terminate
              |
        revoke credentials             <- or the tokens outlive the agents

   the path must not run through the fleet
      via orchestrator API      NO   depends on the thing being stopped
      via the agent sidecar     NO   workload credentials
      out-of-band control plane YES  separate creds, separate network

   tested quarterly, under partial failure, target under 5 minutes
```

D2.4 contained one agent. This is the control for the case where the unit of
containment is the fleet.

The source incident makes the requirement concrete in one detail: third-party
access ended when the **third party** revoked its keys — not when the agents
stopped. Terminating agents while their credentials stay valid leaves the
persistence exactly where it was, and moves the incident rather than ending it.

So the kill switch is one action with four properties:

**One selector.** Experiment, model, time window, or everything. Not a runbook
of forty steps executed under pressure.

**Independent of the agent execution path.** Separate credentials, separate
network path, so a compromised fleet cannot interfere with the thing that stops
it.

**Evidence-preserving.** Snapshot state and transcripts *before* terminating.
An incident you cannot reconstruct afterwards has been survived, not handled.

**Revocation in the same action.** Terminate and revoke together, or the
attacker keeps what the agents were holding.

Plus the operational half: a tested activation path with a measured target
under five minutes, quarterly tests including partial-failure conditions, and a
documented authority to activate that does not require consensus. A kill switch
nobody has pulled is a hypothesis.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">activation path</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">runs through the fleet?</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">credentials</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">usable if the fleet is compromised</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">via the orchestrator API</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">yes</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">shared</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">via the agent&#x27;s own sidecar</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">yes</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the workload&#x27;s</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">out-of-band control plane</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">no</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">separate</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>yes</b></td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">The first two are the ones teams actually build, and both run through the thing they are trying to stop. The switch has to be independent in credentials AND in network path, or it is a request.</div>

## 3 · The procedure, as a skill

Terminating eight agents without revoking leaves eight tokens valid for up to 72 hours. The skill tests both paths, checks that killing preserves the evidence for the incident that triggered it, and measures the whole thing against a target.

### The skill — [`skills/response/fleet-kill-switch-test/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/fleet-kill-switch-test/SKILL.md)

```yaml
name: fleet-kill-switch-test
description: >-
  Test a fleet-wide stop for what it leaves behind — valid tokens, destroyed
  evidence, and the runs that should have been preserved — and measure how long
  the whole thing takes. Use before relying on a kill switch, and after
  designing one.
allowed-tools: Read, Grep, Glob
```

# Terminating the agents leaves every token valid

A fleet kill switch usually terminates workloads. Termination does not revoke
credentials, so every token the fleet held stays valid until it expires — up to
seventy-two hours of an attacker being able to act as agents that no longer
exist. And a kill that does not preserve first destroys the evidence for the
incident that triggered it.

## When to use this

Before a kill switch is relied on, after it is built, and once a year as a
rehearsal.

## Procedure

**1 — Terminate only, and count what stays valid.** Tokens, sessions,
outstanding delegated grants. This is the baseline failure and it is invisible
in a design review.

**2 — Terminate and revoke together, and re-count.** Zero is the target. If
revocation is a separate runbook step performed by a different team, it will not
happen at speed.

**3 — Check evidence preservation.** Kill with and without preservation, then
attempt the reconstruction. A kill switch that destroys the run records is a
containment that ends the investigation.

**4 — Test selectivity.** Stop one agent, one class, the whole fleet. A switch
with only the last setting will not be used until it is far too late, which is
the same as not having one.

**5 — Measure the time.** Decision to last agent stopped and last token revoked.
Set a target and report against it; an untimed rehearsal is a demonstration.

## Example

**Input** — the fixture committed at the top of [`scripts/fleet_kill_switch_test.py`](scripts/fleet_kill_switch_test.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
terminate only        : 8 agents stopped, 8 tokens still valid for up to 72h
terminate and revoke  : 8 agents stopped, 0 tokens still valid

In the source incident, third-party access ended when the third party
revoked its keys - not when the agents stopped. Stopping the process is
the visible half of containment and the smaller one.
terminate first   terminate -> revoke credentials                           reconstructable=False
preserve first    snapshot state and transcripts -> terminate -> revoke credentialsreconstructable=True
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "fleet": {"agents": 0, "tokens": 0},
  "terminate_only": {"tokens_valid_after": 0, "max_validity_hours": 0},
  "terminate_and_revoke": {"tokens_valid_after": 0},
  "preservation": {"preserved": true, "reconstructable": true},
  "selectivity": [{"scope": "one|class|fleet", "supported": true}],
  "timing": {"target_minutes": 0.0, "measured_minutes": 0.0}
}
```

## Failure modes

- **Terminating without revoking.** The credentials outlive the workloads.
- **Killing before preserving.** The incident becomes unreconstructable.
- **A fleet-only switch.** Nobody uses it until the whole estate is on fire.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/fleet-kill-switch-test/scripts/fleet_kill_switch_test.py
SCRIPT = "skills/response/fleet-kill-switch-test/scripts/fleet_kill_switch_test.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Terminating eight agents without revoking leaves all eight tokens valid for up to 72 hours; terminating and revoking together leaves none. Preserving before terminating keeps the incident reconstructable and terminating first does not. Only one of three plausible activation paths survives the fleet being compromised, and of four quarterly tests one was never run and one ran 6.8 minutes against a five-minute target, with the revocation step the part that slowed.

## Your turn

Ask who in your organisation is allowed to stop every agent at once, without asking anyone. If the answer is a committee, you do not have a kill switch — you have an escalation path, and they take different amounts of time.

## Where this leaves you

**What you can do now.** An incident practice for an autonomous actor: reconstruct the timeline with every claim sourced, scope the blast radius from identity and egress logs, contain in seconds rather than hours, a named person with stop authority at 3am, and one tested switch that stops a whole fleet and revokes what it was holding.

**What you still cannot do.** All of it is one incident at a time. Nothing here tells you whether the estate as a whole is governed — how many agents exist, who owns them, which controls apply, and what you would tell a regulator on the Monday.

**Function E is the estate view, and it starts by being precise about a phrase everyone uses loosely. Next → E1.0, what AI governance means.**

---

**Next → [E1.0 · Start here — what AI governance means](https://spbreed.github.io/cyber-commons/lessons/E1.0.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*